# NDI Tip Position Analysis — ICRA 2027
**4-DoF Concentric Tube Robot · Free Space Tests**

Each section corresponds to one *experiment set* (3 repeated trials).
Trajectories are plotted relative to the initial tip position (all traces
start at the origin).  Units: **millimetres** (raw data is in metres).

> **Font note:** plots use *Computer Modern* (the LaTeX default, visually
> identical to Latin Modern Math) via matplotlib's built-in `cm` math font set.
> To enable true Latin Modern / LaTeX rendering, install a LaTeX distribution
> and uncomment the `text.usetex` lines in the Setup cell.


In [ ]:
%matplotlib inline
import os, csv
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D          # noqa: F401
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

# ── Typography ────────────────────────────────────────────────────────────────
# Computer Modern (matplotlib built-in) is visually identical to Latin Modern.
# Uncomment the usetex lines if a LaTeX distribution is installed.
plt.rcParams.update({
    # 'text.usetex': True,            # requires pdflatex / xelatex in PATH
    # 'text.latex.preamble': r'\usepackage{lmodern}',
    'font.family':                 'serif',
    'font.serif':                  ['cmr10', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset':            'cm',     # Computer Modern = Latin Modern Math
    'axes.formatter.use_mathtext': True,
    'font.size':                   9,
    'axes.labelsize':              9,
    'axes.titlesize':              10,
    'xtick.labelsize':             8,
    'ytick.labelsize':             8,
    'legend.fontsize':             8,
    'legend.framealpha':           0.88,
    'legend.edgecolor':            '0.75',
    'figure.dpi':                  100,
    'savefig.dpi':                 300,
    'savefig.bbox':                'tight',
    'savefig.pad_inches':          0.06,
    'lines.linewidth':             1.1,
    'axes.linewidth':              0.65,
    'grid.linewidth':              0.35,
    'grid.alpha':                  0.45,
    'axes.grid':                   True,
})

# ── Constants ─────────────────────────────────────────────────────────────────
FIGURES_DIR = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

# Wong (2011) color-blind-safe palette — distinguishable in B&W print
COLORS = ['#0072B2', '#D55E00', '#009E73']   # blue · vermillion · emerald


In [ ]:
# ── Data loading ──────────────────────────────────────────────────────────────
def load_csv(path):
    """Return (elapsed_s, x_m, y_m, z_m) as NumPy arrays."""
    t, xs, ys, zs = [], [], [], []
    with open(path) as f:
        for row in csv.DictReader(f):
            t.append(float(row['elapsed']))
            xs.append(float(row['x']))
            ys.append(float(row['y']))
            zs.append(float(row['z']))
    return np.array(t), np.array(xs), np.array(ys), np.array(zs)


def remove_outliers(t, x, y, z, jump_mm=2.0):
    """Drop tracker dropout spikes.

    A glitch is a *discontinuity*: the tip teleports and snaps back within one
    sample.  Filtering on absolute position (e.g. a median/MAD z-score) is
    wrong here because the tip legitimately travels tens of millimetres, so
    the extremes of a real trajectory get flagged as outliers and the motion
    is silently truncated.  We instead flag samples whose displacement from
    BOTH neighbours exceeds `jump_mm`, which only fires on true spikes.
    """
    p = np.column_stack([x, y, z]) * 1000.0        # -> mm
    mask = np.ones(len(t), dtype=bool)
    if len(t) >= 3:
        d_prev = np.linalg.norm(p[1:-1] - p[:-2], axis=1)
        d_next = np.linalg.norm(p[2:]  - p[1:-1], axis=1)
        mask[1:-1] = ~((d_prev > jump_mm) & (d_next > jump_mm))
    n_rm = int((~mask).sum())
    return t[mask], x[mask], y[mask], z[mask], n_rm


# ── Geometry helpers ──────────────────────────────────────────────────────────
def fit_plane(x, y, z):
    """SVD best-fit plane through point cloud.
    Returns (centroid, u_axis, v_axis, normal, rms_residual).
    """
    pts = np.column_stack([x, y, z])
    c = pts.mean(0)
    _, _, vt = np.linalg.svd(pts - c, full_matrices=False)
    u, v, n = vt[0], vt[1], vt[2]
    rms = float(np.sqrt(np.mean(((pts - c) @ n) ** 2)))
    return c, u, v, n, rms


def project_plane(x, y, z, centroid, u_ax, v_ax):
    """Project 3-D points onto a plane's local (u, v) coordinate frame."""
    d = np.column_stack([x, y, z]) - centroid
    return d @ u_ax, d @ v_ax


def _equal_3d_aspect(ax, x, y, z):
    """Enforce equal aspect ratio on a 3-D Axes."""
    half = max(max(float(np.ptp(a)) for a in (x, y, z)), 1e-6) / 2.0
    for setter, arr in [(ax.set_xlim, x), (ax.set_ylim, y), (ax.set_zlim, z)]:
        mid = float(arr.max() + arr.min()) / 2.0
        setter(mid - half, mid + half)
    try:
        ax.set_box_aspect((1, 1, 1))
    except Exception:
        pass


In [ ]:
def plot_exp_set(title, config_str, csv_entries, save_name, jump_mm=2.0):
    """
    Build a 6-panel figure for one experiment set (3 repeated trials).

    Panels
    ------
    Top row   : 3-D trajectory · XY projection (top view) · XZ projection (side view)
    Bottom row: best-fit plane projection · relative displacement vs. time · statistics
    """
    M = 1000.0          # metres -> millimetres
    units = 'mm'
    alpha, ms = 0.9, 7
    # Repeat trials often coincide to within ~0.2 mm.  Tapering the width
    # lets the lower traces read as a band instead of being hidden by the
    # last one drawn.
    LWS = [2.1, 1.35, 0.75]

    # ── Load & preprocess ────────────────────────────────────────────────────
    runs = []
    for label, fpath in csv_entries:
        t, x, y, z, n_rm = remove_outliers(*load_csv(fpath), jump_mm=jump_mm)
        x = (x - x[0]) * M
        y = (y - y[0]) * M
        z = (z - z[0]) * M
        t = t - t[0]
        runs.append(dict(label=label, t=t, x=x, y=y, z=z, n=len(t), n_rm=n_rm))

    all_x = np.concatenate([r['x'] for r in runs])
    all_y = np.concatenate([r['y'] for r in runs])
    all_z = np.concatenate([r['z'] for r in runs])
    centroid, u_ax, v_ax, normal, rms = fit_plane(all_x, all_y, all_z)

    # ── Layout ───────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(15, 9.5))
    gs  = GridSpec(2, 3, figure=fig, wspace=0.40, hspace=0.52)
    ax3d  = fig.add_subplot(gs[0, 0], projection='3d')
    ax_xy = fig.add_subplot(gs[0, 1])
    ax_xz = fig.add_subplot(gs[0, 2])
    ax_pl = fig.add_subplot(gs[1, 0])
    ax_dt = fig.add_subplot(gs[1, 1])
    ax_st = fig.add_subplot(gs[1, 2])

    fig.suptitle(f'{title}\n{config_str}', fontsize=11, y=0.998, va='top')

    # ── 3-D trajectory ───────────────────────────────────────────────────────
    for r, c, w in zip(runs, COLORS, LWS):
        ax3d.plot(r['x'], r['y'], r['z'], color=c, lw=w, alpha=alpha,
                  label=r['label'])
        ax3d.scatter([r['x'][0]],  [r['y'][0]],  [r['z'][0]],
                     marker='o', s=55, facecolors='white', edgecolors=c,
                     linewidths=1.6, zorder=6)
        ax3d.scatter([r['x'][-1]], [r['y'][-1]], [r['z'][-1]],
                     marker='s', s=50, color=c, zorder=6)
    ax3d.set_xlabel(f'$\\Delta X$ ({units})', labelpad=3)
    ax3d.set_ylabel(f'$\\Delta Y$ ({units})', labelpad=3)
    ax3d.set_zlabel(f'$\\Delta Z$ ({units})', labelpad=3)
    ax3d.set_title('3D Trajectory', pad=5)
    _equal_3d_aspect(ax3d, all_x, all_y, all_z)
    ax3d.view_init(elev=22, azim=-58)
    ax3d.tick_params(labelsize=7)
    try:
        for pane in [ax3d.xaxis.pane, ax3d.yaxis.pane, ax3d.zaxis.pane]:
            pane.fill = False
            pane.set_edgecolor('0.85')
        ax3d.xaxis.pane.set_alpha(0.0)
        ax3d.yaxis.pane.set_alpha(0.0)
        ax3d.zaxis.pane.set_alpha(0.0)
    except Exception:
        pass
    run_h = [Line2D([0],[0], color=c, lw=w, label=r['label'])
             for r, c, w in zip(runs, COLORS, LWS)]
    sym_h = [Line2D([0],[0], ls='none', marker='o', ms=6, mfc='w', mec='k',
                    mew=1.2, label='Start'),
             Line2D([0],[0], ls='none', marker='s', ms=5, color='k',
                    label='End')]
    ax3d.legend(handles=run_h + sym_h, fontsize=7, loc='upper left',
                framealpha=0.88, edgecolor='0.75')

    # ── XY projection ─────────────────────────────────────────────────────────
    for r, c, w in zip(runs, COLORS, LWS):
        ax_xy.plot(r['x'], r['y'], color=c, lw=w, alpha=alpha)
        ax_xy.plot(r['x'][0],  r['y'][0],  'o', ms=ms, mfc='white',
                   mec=c, mew=1.5, zorder=5)
        ax_xy.plot(r['x'][-1], r['y'][-1], 's', ms=ms-1, color=c, zorder=5)
    ax_xy.set_xlabel(f'$\\Delta X$ ({units})')
    ax_xy.set_ylabel(f'$\\Delta Y$ ({units})')
    ax_xy.set_title('XY Projection (Top View)', pad=5)
    ax_xy.set_aspect('equal', adjustable='datalim')
    ax_xy.spines['top'].set_visible(False)
    ax_xy.spines['right'].set_visible(False)

    # ── XZ projection ─────────────────────────────────────────────────────────
    for r, c, w in zip(runs, COLORS, LWS):
        ax_xz.plot(r['x'], r['z'], color=c, lw=w, alpha=alpha)
        ax_xz.plot(r['x'][0],  r['z'][0],  'o', ms=ms, mfc='white',
                   mec=c, mew=1.5, zorder=5)
        ax_xz.plot(r['x'][-1], r['z'][-1], 's', ms=ms-1, color=c, zorder=5)
    ax_xz.set_xlabel(f'$\\Delta X$ ({units})')
    ax_xz.set_ylabel(f'$\\Delta Z$ ({units})')
    ax_xz.set_title('XZ Projection (Side View)', pad=5)
    ax_xz.set_aspect('equal', adjustable='datalim')
    ax_xz.spines['top'].set_visible(False)
    ax_xz.spines['right'].set_visible(False)

    # ── best-fit plane projection ─────────────────────────────────────────────
    for r, c, w in zip(runs, COLORS, LWS):
        u, v = project_plane(r['x'], r['y'], r['z'], centroid, u_ax, v_ax)
        ax_pl.plot(u, v, color=c, lw=w, alpha=alpha, label=r['label'])
        ax_pl.plot(u[0],  v[0],  'o', ms=ms, mfc='white', mec=c, mew=1.5, zorder=5)
        ax_pl.plot(u[-1], v[-1], 's', ms=ms-1, color=c, zorder=5)
    ax_pl.set_xlabel(f'$u$ ({units})')
    ax_pl.set_ylabel(f'$v$ ({units})')
    ax_pl.set_title(f'Best-Fit Plane (RMS = {rms:.3f} {units})', pad=5)
    ax_pl.set_aspect('equal', adjustable='datalim')
    ax_pl.legend(fontsize=7, loc='best', framealpha=0.88, edgecolor='0.75')
    ax_pl.spines['top'].set_visible(False)
    ax_pl.spines['right'].set_visible(False)

    # ── relative displacement vs. time ────────────────────────────────────────
    for r, c, w in zip(runs, COLORS, LWS):
        ax_dt.plot(r['t'], r['x'], '-',  color=c, lw=w*0.8, alpha=alpha)
        ax_dt.plot(r['t'], r['y'], '--', color=c, lw=w*0.8, alpha=alpha)
        ax_dt.plot(r['t'], r['z'], ':',  color=c, lw=w*0.8, alpha=alpha)
    ax_dt.axhline(0, color='0.55', lw=0.45, ls='--', zorder=0)
    ax_dt.set_xlabel('Elapsed Time (s)')
    ax_dt.set_ylabel(f'Relative Displacement ({units})')
    ax_dt.set_title('Position vs. Time', pad=5)
    ax_handles = [
        Line2D([0],[0], color='0.35', ls='-',  lw=1, label='$\\Delta X$'),
        Line2D([0],[0], color='0.35', ls='--', lw=1, label='$\\Delta Y$'),
        Line2D([0],[0], color='0.35', ls=':',  lw=1, label='$\\Delta Z$'),
    ]
    ax_dt.legend(handles=ax_handles, fontsize=7, loc='upper left',
                 framealpha=0.88, edgecolor='0.75')
    ax_dt.spines['top'].set_visible(False)
    ax_dt.spines['right'].set_visible(False)

    # ── statistics panel ──────────────────────────────────────────────────────
    fx = np.array([r['x'][-1] for r in runs])
    fy = np.array([r['y'][-1] for r in runs])
    fz = np.array([r['z'][-1] for r in runs])
    mean_f  = np.array([fx.mean(), fy.mean(), fz.mean()])
    finals  = np.column_stack([fx, fy, fz])
    errs_3d = np.linalg.norm(finals - mean_f, axis=1)

    sep = '-' * 32
    txt = (
        f'Final Position Statistics\n{sep}\n'
        f'  dX: {fx.mean():+7.3f} ± {fx.std():.4f} {units}\n'
        f'  dY: {fy.mean():+7.3f} ± {fy.std():.4f} {units}\n'
        f'  dZ: {fz.mean():+7.3f} ± {fz.std():.4f} {units}\n'
        f'{sep}\n'
        f'  3D Repeatability:\n'
        f'    mean = {errs_3d.mean():.4f} {units}\n'
        f'    max  = {errs_3d.max():.4f} {units}\n'
        f'{sep}\n'
        f'  Plane RMS = {rms:.4f} {units}\n'
        f'{sep}\n'
    )
    for r in runs:
        rm = f'  (-{r["n_rm"]} outliers)' if r['n_rm'] else ''
        txt += f'  {r["label"]}: {r["n"]:d} pts{rm}\n'

    ax_st.axis('off')
    ax_st.text(0.04, 0.97, txt, transform=ax_st.transAxes,
               fontsize=7.5, va='top', ha='left', fontfamily='monospace',
               bbox=dict(boxstyle='round,pad=0.55', facecolor='#f5f5f5',
                         edgecolor='0.78', lw=0.8))
    ax_st.set_title('Statistics', pad=5)

    # ── save ─────────────────────────────────────────────────────────────────
    out_path = os.path.join(FIGURES_DIR, f'{save_name}.jpg')
    fig.savefig(out_path, dpi=300, bbox_inches='tight',
                pil_kwargs={'quality': 95, 'optimize': True})
    print(f'  Saved -> {out_path}')
    plt.show()
    return fig


## 1st Free Space Test (Exp 1–3)
**Configuration:** OTT = 20 mm, ITT = 20 mm $|$ OTR = 0$^\circ$, ITR = 0$^\circ$  
Both tubes translate 20 mm forward with no rotation (pure straight extension).

In [ ]:
plot_exp_set(
    title='1st Free Space Test',
    config_str='OTT = 20 mm,  ITT = 20 mm  $|$  OTR = 0$^\\circ$,  ITR = 0$^\\circ$',
    csv_entries=[
        ('Exp 1', 'exp1_20260811_215059.csv'),
        ('Exp 2', 'exp2_20260811_215200.csv'),
        ('Exp 3', 'exp3_20260811_215253.csv'),
    ],
    save_name='set1_free_space_1',
)

## 2nd Free Space Test (Exp 5–7)
**Configuration:** OTT = 0 mm, ITT = 35 mm $|$ OTR = 0$^\circ$, ITR = 360$^\circ$  
Inner tube extends 35 mm and rotates a full revolution; outer tube is stationary.

In [ ]:
plot_exp_set(
    title='2nd Free Space Test',
    config_str='OTT = 0 mm,  ITT = 35 mm  $|$  OTR = 0$^\\circ$,  ITR = 360$^\\circ$  (no outer tube movement)',
    csv_entries=[
        ('Exp 5', 'exp5_20260811_222950.csv'),
        ('Exp 6', 'exp6_20260811_223714.csv'),
        ('Exp 7', 'exp7_20260811_224018.csv'),
    ],
    save_name='set2_free_space_2',
)

## 3rd Free Space Test — Top Right (Exp 8–10)
**Configuration:** OTT = 35 mm, ITT = 35 mm $|$ OTR = 360$^\circ$, ITR = 0$^\circ$  
Both tubes extend 35 mm; outer tube makes a full rotation while inner stays fixed.

*Note:* two files exist for Exp 8; `...225935.csv` is a 5.6 s aborted run
(0.05 mm net motion), so `...230128.csv` is used.

In [ ]:
plot_exp_set(
    title='3rd Free Space Test - Top Right',
    config_str='OTT = 35 mm,  ITT = 35 mm  $|$  OTR = 360$^\\circ$,  ITR = 0$^\\circ$',
    csv_entries=[
        ('Exp 8', 'exp8_20260811_230128.csv'),
        ('Exp 9', 'exp9_20260811_230810.csv'),
        ('Exp 10', 'exp10_20260811_231038.csv'),
    ],
    save_name='set3a_free_space_3_top_right',
)

## 3rd Free Space Test — Bottom Left (Exp 11–13)
**Configuration:** OTT = 35 mm, ITT = 35 mm $|$ OTR = 360$^\circ$, ITR = 360$^\circ$  
Both tubes extend 35 mm; both rotate a full revolution simultaneously.

In [ ]:
plot_exp_set(
    title='3rd Free Space Test - Bottom Left',
    config_str='OTT = 35 mm,  ITT = 35 mm  $|$  OTR = 360$^\\circ$,  ITR = 360$^\\circ$',
    csv_entries=[
        ('Exp 11', 'exp11_20260811_232209.csv'),
        ('Exp 12', 'exp12_20260811_232456.csv'),
        ('Exp 13', 'exp13_20260811_232803.csv'),
    ],
    save_name='set3b_free_space_3_bottom_left',
)

## 4th Free Space Test — Bottom Mid (Exp 14–16)
**Configuration:**  
- *Step 1:* OTT = 35 mm, ITT = 35 mm  
- *Step 2:* OTR = 0$^\circ$, ITR = 180$^\circ$ (half rotation)  
- *Step 3:* ITT += 35 mm (additional 35 mm inner extension)  

Three-phase motion: extend → half-rotate inner → extend further.

In [ ]:
plot_exp_set(
    title='4th Free Space Test - Bottom Mid',
    config_str='Step 1: OTT = 35 mm, ITT = 35 mm  $|$  Step 2: OTR = 0$^\\circ$, ITR = 180$^\\circ$  $|$  Step 3: ITT += 35 mm',
    csv_entries=[
        ('Exp 14', 'exp14_20260811_233834.csv'),
        ('Exp 15', 'exp15_20260811_234353.csv'),
        ('Exp 16', 'exp16_20260811_235022.csv'),
    ],
    save_name='set4a_free_space_4_bottom_mid',
)

## 4th Free Space Test — Bottom Right (Exp 17–19)
**Configuration:**  
- *Step 1:* OTT = 17.5 mm, ITT = 17.5 mm  
- *Step 2:* OTR = 0$^\circ$, ITR = 180$^\circ$ (half rotation)  
- *Step 3:* ITT += 17.5 mm  

Same sequence as Bottom Mid but at half the translation amplitude.

*Note:* two files exist for Exp 18; `...010107.csv` contains only a header,
so `...010331.csv` is used.  `exp17 copy.csv` is byte-identical to `exp17.csv`.

In [ ]:
plot_exp_set(
    title='4th Free Space Test - Bottom Right',
    config_str='Step 1: OTT = 17.5 mm, ITT = 17.5 mm  $|$  Step 2: OTR = 0$^\\circ$, ITR = 180$^\\circ$  $|$  Step 3: ITT += 17.5 mm',
    csv_entries=[
        ('Exp 17', 'exp17_20260812_001514.csv'),
        ('Exp 18', 'exp18_20260812_010331.csv'),
        ('Exp 19', 'exp19_20260812_010534.csv'),
    ],
    save_name='set4b_free_space_4_bottom_right',
)